# Task 6: EV Charging Demand Analysis and Prediction
**Prepared by:** Monika  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

## Project objective
Explore electric-vehicle charging-session records, assess data quality, investigate demand patterns, and build a regression baseline to estimate the continuous `Demand` variable.

> **Important:** Place the original dataset named `ChargingRecords.csv` in the same folder as this notebook before running it. This notebook does not fabricate the source dataset. All results and plots are computed from the file you provide.

## 1. Import libraries
Run this cell first. If a package is missing, install it using `pip install pandas numpy matplotlib seaborn scikit-learn`. 

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

## 2. Load the dataset

In [ ]:
DATA_PATH = "ChargingRecords.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "ChargingRecords.csv was not found. Put the original CSV in the same folder as this notebook, "
        "or update DATA_PATH to the correct file location."
    )

df_raw = pd.read_csv(DATA_PATH)
print("Dataset shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())

## 3. Data quality audit

In [ ]:
print("Rows and columns:", df_raw.shape)
print("\\nColumn names and data types:")
display(pd.DataFrame({"dtype": df_raw.dtypes.astype(str), "missing_count": df_raw.isna().sum(),
                      "missing_percent": (df_raw.isna().mean() * 100).round(2)}))
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))

## 4. Cleaning and feature engineering
The rules below are intentionally transparent. Negative durations are treated as invalid. Duration outlier filtering is optional because unusually long sessions can be real; compare results with and without filtering before making operational decisions.

In [ ]:
df = df_raw.copy()

# Remove exact duplicate records
before = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", before - len(df))

# Parse available datetime columns and derive time features
for col in ["StartDatetime", "EndDatetime"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "StartDatetime" in df.columns:
    df["StartHour"] = df["StartDatetime"].dt.hour
    df["StartDayofWeek"] = df["StartDatetime"].dt.day_name()
    df["StartMonth"] = df["StartDatetime"].dt.month

# If a duration column exists, remove physically invalid negative durations
if "Duration" in df.columns:
    df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")
    invalid_duration = df["Duration"] < 0
    print("Negative-duration rows removed:", int(invalid_duration.sum()))
    df = df.loc[~invalid_duration].copy()

if "Demand" not in df.columns:
    raise ValueError("The dataset must contain a 'Demand' target column.")

df["Demand"] = pd.to_numeric(df["Demand"], errors="coerce")
df = df.dropna(subset=["Demand"]).copy()

print("Shape after basic cleaning:", df.shape)
display(df.head())

## 5. Descriptive statistics

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
display(numeric_df.describe().T)

if "Demand" in df.columns:
    print("Demand mean:", round(df["Demand"].mean(), 4))
    print("Demand median:", round(df["Demand"].median(), 4))
    print("Demand standard deviation:", round(df["Demand"].std(), 4))
    print("Demand minimum:", round(df["Demand"].min(), 4))
    print("Demand maximum:", round(df["Demand"].max(), 4))

## 6. Visual exploration

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["Demand"], kde=True)
plt.title("Distribution of Charging Demand")
plt.xlabel("Demand")
plt.ylabel("Number of sessions")
plt.tight_layout()
plt.show()

# Explain the shape in the report using the computed mean, median, skewness, and visible distribution.
print("Demand skewness:", round(df["Demand"].skew(), 4))
print("Interpretation guide: positive skewness indicates a longer right tail; negative skewness indicates a longer left tail.")

In [ ]:
if "StartHour" in df.columns:
    hourly = df.groupby("StartHour", dropna=True)["Demand"].agg(["mean", "median", "count"]).reset_index()
    display(hourly.sort_values("mean", ascending=False))
    plt.figure(figsize=(9, 4))
    sns.lineplot(data=hourly, x="StartHour", y="mean", marker="o")
    plt.title("Average Demand by Start Hour")
    plt.xlabel("Start hour (0–23)")
    plt.ylabel("Average demand")
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.show()
else:
    print("StartDatetime is unavailable; hourly analysis was skipped.")

In [ ]:
if len(numeric_df.columns) >= 2:
    corr = numeric_df.corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
    plt.title("Correlation Heatmap for Numeric Variables")
    plt.tight_layout()
    plt.show()
    if "Demand" in corr.columns:
        display(corr["Demand"].sort_values(ascending=False).to_frame("correlation_with_Demand"))
else:
    print("Not enough numeric columns for a correlation heatmap.")

## 7. Prepare a regression dataset
The target is `Demand`. The notebook uses available predictors from a shortlist. Categorical columns are one-hot encoded inside a scikit-learn pipeline to avoid treating category labels as ordered numeric values. Identifier columns are excluded.

In [ ]:
candidate_features = [
    "ChargerType", "ChargerCompany", "Duration", "StartHour", "StartMonth",
    "StartDayofWeek", "Location"
]
feature_cols = [c for c in candidate_features if c in df.columns]

if not feature_cols:
    raise ValueError(
        "No supported predictor columns were found. Update candidate_features to match your dataset columns."
    )

X = df[feature_cols].copy()
y = df["Demand"].copy()

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Features used:", feature_cols)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 8. Train and evaluate Random Forest regression

In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1,
        min_samples_leaf=2
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Value": [mae, rmse, r2],
    "Interpretation": [
        "Average absolute prediction error in target units; lower is better.",
        "Penalizes larger errors more strongly; lower is better.",
        "Relative to predicting the test mean; closer to 1 is generally better."
    ]
})
display(metrics)
print("Do not report these values until this notebook has been run on the intended dataset.")

## 9. Actual vs predicted and residual analysis

In [ ]:
results = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": pred})
results["Residual"] = results["Actual"] - results["Predicted"]
display(results.head(10))

plt.figure(figsize=(6, 6))
sns.scatterplot(data=results, x="Actual", y="Predicted", alpha=0.5)
low = min(results["Actual"].min(), results["Predicted"].min())
high = max(results["Actual"].max(), results["Predicted"].max())
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Actual vs Predicted Demand")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(results["Residual"], kde=True)
plt.title("Distribution of Prediction Residuals")
plt.xlabel("Residual (actual − predicted)")
plt.tight_layout()
plt.show()

## 10. Interpretation checklist
After running the notebook, write interpretations using the displayed values—do not copy example numbers from an earlier draft unless they match this run.

- Report dataset dimensions, missingness, duplicates, and every cleaning decision.
- Describe the Demand distribution using mean, median, spread, and skewness.
- Quote the actual highest and lowest hourly average only if hourly results are available.
- Explain selected correlation coefficients and note that correlation does not prove causation.
- Report MAE, RMSE, and R² from the current run, including their units and limitations.
- Discuss errors by time, location, or charger category if sample sizes permit.
- For stronger future evaluation, use time-based validation when predicting future periods.
- Treat feature importance as model association, not causal contribution.

## 11. Limitations
This analysis depends on the provided dataset's coverage, variable definitions, data quality, and representativeness. A random train/test split may overstate future performance when observations from similar time periods appear in both sets. Outlier removal can exclude valid unusual sessions. Missing operational variables can limit accuracy. External validation is needed before using predictions for capacity or infrastructure decisions.

## 12. Conclusion
Use the computed results to write a concise conclusion describing the main demand patterns, model performance, important caveats, and next steps. Do not claim the model is deployment-ready without time-based validation and monitoring.